In [5]:
import duckdb



In [11]:
con = duckdb.connect(database='dados_duckdb.db', read_only=False)

In [16]:
df = con.execute("""
            SELECT *
            FROM(
                SELECT *, ROW_NUMBER() OVER (PARTITION BY NATBR ORDER BY data_ingestao DESC) as linha
                FROM bronze_z0019
                WHERE data_ingestao >= '2026-06-10'
            ) WHERE linha = 1
""").fetchdf()

df.head(10)

,NATBR,MARTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao,linha
0,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-06-10 00:10:14.622400,1
1,10002,MARTELO,BT50,100,1500,z0019_1.csv,2026-06-10 00:10:14.622400,1
2,10004,SERRA,BT50,100,200,z0019_2.csv,2026-06-10 00:17:22.656826,1
3,10005,MACHADO,BT10,100,100,z0019_2.csv,2026-06-10 00:17:22.656826,1
4,10003,PREGO,BT10,100,60,z0019_2.csv,2026-06-10 00:17:22.656826,1


In [21]:
df_final = df.drop(columns=['nome_arquivo','data_ingestao','linha'])
df_final = df_final.rename(columns={"NATBR":"id"})
df_final = df_final.rename(columns={"MARTX":"nm_produto"})
df_final = df_final.rename(columns={"WERKS":"id_categoria"})
df_final = df_final.rename(columns={"MAINS":"id_fornecedor"})
df_final = df_final.rename(columns={"LABST":"vl_preco"})
df_final.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10001,PARAFUSO,BT10,100,100
1,10002,MARTELO,BT50,100,1500
2,10004,SERRA,BT50,100,200
3,10005,MACHADO,BT10,100,100
4,10003,PREGO,BT10,100,60


In [29]:
df2= df_final
df2 = df2.astype(
    {
        'id': int
        ,'nm_produto': str
        ,'id_categoria': str
        ,'id_fornecedor':int
        ,'vl_preco': float
    }
)
#df2.dtypes
df2.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10001,PARAFUSO,BT10,100,100.0
1,10002,MARTELO,BT50,100,1500.0
2,10004,SERRA,BT50,100,200.0
3,10005,MACHADO,BT10,100,100.0
4,10003,PREGO,BT10,100,60.0


In [32]:
con.execute("""
    CREATE TABLE IF NOT EXISTS produtos (
            id BIGINT
            ,nm_produto TEXT
            ,id_categoria TEXT
            ,id_fornecedor BIGINT
            ,vl_preco FLOAT
            )
        """)
con.execute ("INSERT INTO produtos SELECT * FROM df2")

In [31]:
df2.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10001,PARAFUSO,BT10,100,100.0
1,10002,MARTELO,BT50,100,1500.0
2,10004,SERRA,BT50,100,200.0
3,10005,MACHADO,BT10,100,100.0
4,10003,PREGO,BT10,100,60.0


In [35]:
df_resultado = con.execute("SELECT * FROM produtos").fetchdf()
df_resultado.head(10)

ConnectionException: Connection Error: Connection already closed!

In [34]:
con.close()